In [1]:
# importing necessary libraries
import numpy as np
import pandas as pd


In [2]:

# ============================================================
# STEP 1: LOAD THE DATASET
# ============================================================

#Read the CSV from the data folder
df = pd.read_csv("data/raw/responsivenes.csv")

In [3]:
# ============================================================
# STEP 2: STANDARDIZE COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(".", "_", regex=False)   # eye.opening -> eye_opening
    .str.replace(" ", "_", regex=False)
)

# GCS column expand: "gcs" is a short form, so rename it to
# the full term so anyone reading the data understands it.
# Glasgow Coma Scale: range 3-15
# Also rename inpatient_number to patient_id

df = df.rename(columns={
    "gcs": "glasgow_coma_scale",
    "inpatient_number": "patient_id"
})

print("\nColumns:", list(df.columns))



Columns: ['patient_id', 'eye_opening', 'verbal_response', 'movement', 'consciousness', 'glasgow_coma_scale']


In [4]:
# ============================================================
# STEP 3: MISSING VALUES AND DUPLICATES
# ============================================================

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate patient IDs:", df["patient_id"].duplicated().sum())

df = df.drop_duplicates()


Missing values:
patient_id            0
eye_opening           0
verbal_response       0
movement              0
consciousness         0
glasgow_coma_scale    0
dtype: int64

Duplicate rows: 0
Duplicate patient IDs: 0


In [5]:
# ============================================================
# STEP 4: CLEAN CONSCIOUSNESS TEXT
# ============================================================
# Only strip spaces. Original values are kept so they still
# match the data dictionary and teammates' files.
# (.str.strip() keeps real missing values as NaN, not "nan")
 
df["consciousness"] = df["consciousness"].str.strip()
 
print("\nConsciousness values:")
print(df["consciousness"].value_counts(dropna=False).to_string())
 


Consciousness values:
consciousness
Clear                1974
ResponsiveToSound      19
Nonresponsive          11
ResponsiveToPain        4


In [6]:
# ============================================================
# STEP 5: CONVERT GLASGOW COMA SCALE COLUMNS TO NUMBERS
# ============================================================

components = ["eye_opening", "verbal_response", "movement"]

for column in components + ["glasgow_coma_scale"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")


In [7]:
# ============================================================
# STEP 6: VALIDATE RANGES
# ============================================================
# Eye 1-4, Verbal 1-5, Movement 1-6, Glasgow Coma Scale 3-15
# Out-of-range values become NaN.

valid_ranges = {
    "eye_opening": (1, 4),
    "verbal_response": (1, 5),
    "movement": (1, 6),
    "glasgow_coma_scale": (3, 15),
}

for column, (low, high) in valid_ranges.items():
    invalid = df[column].notna() & ~df[column].between(low, high)
    print(f"Out of range in {column}: {invalid.sum()}")
    df.loc[invalid, column] = np.nan
 

Out of range in eye_opening: 0
Out of range in verbal_response: 0
Out of range in movement: 0
Out of range in glasgow_coma_scale: 0


In [8]:
# ============================================================
# STEP 7: VERIFY AND FIX GLASGOW COMA SCALE TOTAL
# ============================================================
# Glasgow Coma Scale = eye + verbal + movement.
# Only replace the total when all three components are valid,
# so a good stored total is never overwritten with NaN.
# Total must equal eye + verbal + movement - recalculate only when all 3 parts are valid (0 mismatches found in this file)

all_valid = df[components].notna().all(axis=1)
calculated_total = df[components].sum(axis=1)

mismatch = all_valid & (df["glasgow_coma_scale"] != calculated_total)
print("\nGlasgow Coma Scale mismatches fixed:", mismatch.sum())

df.loc[all_valid, "glasgow_coma_scale"] = calculated_total[all_valid]

# Use nullable integers so scores stay whole numbers
for column in components + ["glasgow_coma_scale"]:
    df[column] = df[column].astype("Int64")



Glasgow Coma Scale mismatches fixed: 0


Reasoning: Why flag Clear consciousness vs low Glasgow Coma Scale?
Both columns describe how awake the patient is. A score of 8 or lower means severe impairment (coma level), so it shouldn't appear with "Clear" consciousness. 3 patients have this conflict. We flag them instead of deleting or fixing them, because we can't tell which value is wrong. It could be an entry error, the two being recorded at different times, or sedation. The rows are kept, and the flag lets the team include or exclude them in the analysis.

In [9]:
# ============================================================
# STEP 8: FLAG CONSCIOUSNESS vs GLASGOW COMA SCALE MISMATCH
# ============================================================
# "Clear" consciousness with Glasgow Coma Scale <= 8 doesn't match.
# Rows are kept, just flagged for review.

df["consciousness_mismatch_flag"] = (
    (df["consciousness"] == "Clear") & (df["glasgow_coma_scale"] <= 8)
).fillna(False)

print("\nFlagged rows (Clear but Glasgow Coma Scale <= 8):", df["consciousness_mismatch_flag"].sum())
print(df[df["consciousness_mismatch_flag"]])


Flagged rows (Clear but Glasgow Coma Scale <= 8): 3
     patient_id  eye_opening  verbal_response  movement consciousness  \
424      838076            4                2         1         Clear   
487      731880            4                2         1         Clear   
626      781960            1                1         1         Clear   

     glasgow_coma_scale  consciousness_mismatch_flag  
424                   7                         True  
487                   7                         True  
626                   3                         True  


In [11]:

# ============================================================
# STEP 9: FINAL CHECK AND SAVE
# ============================================================

print("\nFinal shape:", df.shape)
print(df.head())

df.to_csv("data/cleaned/responsivenes_cleaned.csv", index=False)

print("\nCleaning completed successfully.")
print("Final dataset shape:", df.shape)


Final shape: (2008, 7)
   patient_id  eye_opening  verbal_response  movement consciousness  \
0      857781            4                5         6         Clear   
1      743087            4                5         6         Clear   
2      866418            4                5         6         Clear   
3      775928            4                5         6         Clear   
4      810128            4                5         6         Clear   

   glasgow_coma_scale  consciousness_mismatch_flag  
0                  15                        False  
1                  15                        False  
2                  15                        False  
3                  15                        False  
4                  15                        False  

Cleaning completed successfully.
Final dataset shape: (2008, 7)
